# LangChain Memory


<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-memory-202503.ipynb
</div>

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>

## Environment Setup

<br>

> 💡 **Important:**
>
> This notebook requires **LangChain < 0.3**.
>
> Below, you will find two options for installing the required dependencies. Choose the one that best matches your environment::
>
> 🖥️ **Running locally?**  
> → Use **Option A** to create a dedicated virtual environment (recommended).
>
> ☁️ **Using Google Colab or want a quick setup?**  
> → Use **Option B** to install the required dependencies directly.
>

<br><br>


### Option A: Use a virtual environment (recommended)

To create a virtual environment, follow the steps in this document:
- https://gist.github.com/luisjunco/6ade3de82b7804f3ebf912866b4efff5

For this notebook, you'll need these dependencies:

```bash
python -m pip install \
    "langchain<0.3" \
    "langchain-core<0.3" \
    "langchain-community<0.3" \
    "langchain-openai<0.2" \
    ipykernel
```


### Option B: Install dependencies directly

If you are using Google Colab, or you're having problems configuring a virtual environment, create a code cell and run the command below:

```python
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"
```

<br>


</div>

<br>

## Check LangChain version

For this notebook, you'll need LangChain 0.2.x (e.g., 0.2.17)

To check your LangChain version, you can run the command below:

In [1]:
!pip show langchain

Name: langchain
Version: 0.2.17
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: /Users/luis/Desktop/ironhack_sept26/demos_ai_engineering_sept2026/.venv/langchain-v0.2.x/lib/python3.11/site-packages
Requires: aiohttp, langchain-core, langchain-text-splitters, langsmith, numpy, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: langchain-community


<br>

## Intro to Memory



Most common use-case for building an AI application with LangChain is a chatbot trained on a knowledgebase. But there is a problem with just using a RetrievalQA method i.e the chatbot developing using this method has no memory and thus cannot answer in a conversational manner

For example if you check the below flow

**Human** : How to add a video funnel

**AI**    : If you want to set up a video funnel, these are the steps you need to follow:

Step 1- First of all, go to the Marketing tab at the video level.

Step 2- Click on the “Add new funnel” button.

Step 3- Set the title to grab the users’ attention.

Step 4- Set the text for the action that you want would be taken by the users.

Step 5- Select the video for the Funnel.

Step 6- Set the start time at which the pop-up would appear to the user.

Step 7- Once you are satisfied with all the things, click on the “Save Changes” button.

**Human** : Now tell only 5 steps from above

**AI**.   : Sorry I am unable to answer, can you provide more context

As you can see, the chatbot has no memory and thus unable to answer any follow up questions. To solve this we will be using Memory


The memory allows a Large Language Model (LLM) to remember previous interactions with the user. By default, LLMs are stateless — meaning each incoming query is processed independently of other interactions. The only thing that exists for a stateless agent is the current input, nothing else.

There are many applications where remembering previous interactions is very important, such as chatbots. Conversational memory allows us to do that.

There are several ways that we can implement conversational memory. In the context of LangChain, they are all built on top of the `ConversationChain`.

### Types of Memory

Let's discuss some of the most popular memory methods available in Langchain.

1. ConversationBufferMemory
2. ConversationBufferWindowMemory
3. ConversationTokenBufferMemory
4. ConversationSummaryMemory

We shall discuss each of these with an example

### ConversationBufferMemory

This is a simple method that involves storing every chat interaction directly in the buffer. Although it provides good results, it has few drawbacks

1. Because every message is stored, the amount of data being sent to api is high and thus results in higher costs and slower speed of response
2. ChatGPT has input context limit which can get crossed with few messages and thus can result in error

Let's understand it's working with the help of an example. We will try adding memory to a chain

In [2]:
from langchain.llms import OpenAI
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory, ConversationBufferWindowMemory, ConversationSummaryBufferMemory

from langchain.callbacks import get_openai_callback

import os

In [3]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [4]:
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model='gpt-6-luna',
    temperature=1,  # gpt-6-luna only accepts the default temperature
)


conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory()
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_2251/263910356.py:8: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html instead.
  conversation = ConversationChain(


We can see the prompt template used by the `ConversationChain` like so:

In [5]:
print(conversation.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


Here, the prompt primes the model by telling it that the following is a conversation between a human (us) and an AI (`gpt3-turbo`). The prompt attempts to reduce hallucinations (where a model makes things up) by stating:

**"If the AI does not know the answer to a question, it truthfully says it does not know."**

This can help but does not solve the problem of hallucinations — but we will save this for the topic of a future chapter.

Following the initial prompt, we see two parameters; `{history}` and `{input}`. The `{input}` is where we’d place the latest human query; 

The `{history}` is where conversational memory is used. Here, we feed in information about the conversation history between the human and AI.

These two parameters — `{history}` and `{input}` — are passed to the LLM within the prompt template we just saw, and the output that we (hopefully) return is simply the predicted continuation of the conversation.



In [6]:
conversation_buf = ConversationChain(
    llm=llm,
    memory=ConversationBufferMemory()
)

In [7]:
conversation_buf.invoke("Good morning AI!")

{'input': 'Good morning AI!',
 'history': '',
 'response': 'Good morning! I hope your day is off to a great start. What can I help you with today?'}

We return the first response from the conversational agent. Let’s continue the conversation, writing prompts that the LLM can only answer if it considers the conversation history. We also add a count_tokens function so we can see how many tokens are being used by each interaction.

In [8]:
def count_tokens(chain, query):
    with get_openai_callback() as cb:
        result = chain.run(query)
        print(f'Spent a total of {cb.total_tokens} tokens')

    return result

In [9]:
count_tokens(
    conversation_buf, 
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_2251/1595156071.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result = chain.run(query)


Spent a total of 415 tokens


'That’s a fascinating area. An LLM can be paired with external knowledge in a few different ways, depending on whether the knowledge is changing, how reliable answers need to be, and how much control you want over what the model can access.\n\nA common approach is **retrieval-augmented generation (RAG)**: documents are indexed, a user’s question is used to find relevant passages, and those passages are provided to the model as context for its answer. This can make answers more current and easier to ground in sources, without retraining the model whenever the documents change.\n\nOther options include letting the model call **APIs or tools**—for example, to query a database, check current information, or perform a calculation—or **fine-tuning** it for a particular task or style. Fine-tuning can teach behavior and patterns, but it’s usually not the best way to keep a large, frequently changing collection of facts up to date.\n\nThe interesting challenges are often less about connecting a

In [10]:
count_tokens(
    conversation_buf,
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 1338 tokens


'There are several ways to connect an LLM with external knowledge. They differ in **when the information is added**, **how the model accesses it**, and **how much control you need**.\n\n### 1. Put the knowledge directly in the prompt\nProvide a small set of relevant facts or documents alongside each question.\n\n- **Good for:** Short reference material, prototypes, or instructions that rarely change.\n- **Advantages:** Simple to build and easy to inspect.\n- **Limitations:** The prompt has a size limit, and supplying lots of irrelevant material can make answers worse or more expensive.\n\n### 2. Retrieval-augmented generation (RAG)\nStore documents in a searchable index. For each question, retrieve likely relevant passages and give them to the model as context.\n\n- **Good for:** Search across manuals, policies, reports, or other collections of documents.\n- **Advantages:** The source material can be updated without retraining the model; answers can cite the passages used.\n- **Limitat

In [11]:
count_tokens(
    conversation_buf, 
    "What is my aim again?"
)

Spent a total of 1323 tokens


'Your aim is to explore the potential of integrating large language models with external knowledge and analyze the different ways it could be done.'

The LLM can clearly remember the history of the conversation. Let’s take a look at how this conversation history is stored by the ConversationBufferMemory:

In [12]:
print(conversation_buf.memory.buffer)

Human: Good morning AI!
AI: Good morning! I hope your day is off to a great start. What can I help you with today?
Human: My interest here is to explore the potential of integrating Large Language Models with external knowledge
AI: That’s a fascinating area. An LLM can be paired with external knowledge in a few different ways, depending on whether the knowledge is changing, how reliable answers need to be, and how much control you want over what the model can access.

A common approach is **retrieval-augmented generation (RAG)**: documents are indexed, a user’s question is used to find relevant passages, and those passages are provided to the model as context for its answer. This can make answers more current and easier to ground in sources, without retraining the model whenever the documents change.

Other options include letting the model call **APIs or tools**—for example, to query a database, check current information, or perform a calculation—or **fine-tuning** it for a particular

We can see that the buffer saves every interaction in the chat history directly. There are a few pros and cons to this approach. In short, they are:

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid #000;
            padding: 10px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>
    <table>
        <thead>
            <tr>
                <th>Pros</th>
                <th>Cons</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Storing everything gives the LLM the maximum amount of information</td>
                <td>More tokens mean slowing response times and higher costs</td>
            </tr>
            <tr>
                <td>Storing everything is simple and intuitive</td>
                <td>Long conversations may exceed the model's context window, causing earlier information to be truncated or omitted. Although modern models offer much larger context windows, this remains a factor to consider, especially when using models with smaller context windows</td>
            </tr>
        </tbody>
    </table>
</body>
</html>


The `ConversationBufferMemory` is an excellent option to get started with but is limited by the storage of every interaction. Let’s take a look at other options that help remedy this.

### ConversationSummaryMemory
Using ConversationBufferMemory, we very quickly use a lot of tokens and even exceed the context window limit of even the most advanced LLMs available today.

To avoid excessive token usage, we can use ConversationSummaryMemory. As the name would suggest, this form of memory summarizes the conversation history before it is passed to the {history} parameter.

We initialize the ConversationChain with the summary memory like so:

In [13]:
conversation_sum = ConversationChain(
	llm=llm,
	memory=ConversationSummaryMemory(llm=llm)
)

When using `ConversationSummaryMemory`, we need to pass an LLM to the object because the summarization is powered by an LLM. We can see the prompt used to do this here:

In [14]:
print(conversation_sum.memory.prompt.template)

Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:
{summary}

New lines of conversation:
{new_lines}

New summary:


Using this, we can summarize every new interaction and append it to a “running summary” of all past interactions. Let’s have another conversation utilizing this approach.

In [15]:
# without count_tokens we'd call `conversation_sum("Good morning AI!")`
# but let's keep track of our tokens:
count_tokens(
    conversation_sum, 
    "Good morning AI!"
)

Spent a total of 365 tokens


'Good morning! I hope your day is off to a lovely start. What can I help you with today?'

In [16]:
count_tokens(
    conversation_sum, 
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

Spent a total of 1040 tokens


'That’s a rich area to explore. Connecting an LLM to external knowledge can help it answer questions using information that is newer, more specialized, or more specific to an organization than what was present in its training data.\n\nA few common approaches are:\n\n- **Retrieval-augmented generation (RAG):** Search a collection of documents for relevant passages, then give those passages to the model as context for its answer. This is often a practical starting point for private or frequently updated knowledge.\n- **Tool or API use:** Let the model query structured systems—such as a database, search service, or live API—when it needs a particular fact or action.\n- **Knowledge graphs:** Represent entities and relationships explicitly, which can help with questions requiring multi-step connections across facts.\n- **Fine-tuning:** Adapt the model’s behavior or expertise for a task. It’s generally less suited than retrieval for keeping factual knowledge current, since updates require fu

In [17]:
count_tokens(
    conversation_sum, 
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 3052 tokens


'There are several ways to connect an LLM to information beyond what it learned during training. They differ in **when the model gets the information**, **how current it is**, and **how much control you need over evidence and actions**.\n\n### 1. Retrieval-augmented generation (RAG)\nBefore answering, the system searches a collection of documents—such as manuals, policies, research papers, or company files—and gives relevant passages to the model.\n\n- **Good for:** private or specialized information that changes occasionally.\n- **Strengths:** the source material can be updated without retraining; answers can cite documents.\n- **Challenges:** search may retrieve the wrong passages, miss important context, or return conflicting sources.\n- **Example:** an assistant answers questions about a company’s HR policies using the current policy handbook.\n\nRAG can use keyword search, semantic (meaning-based) search, or a combination. It often works better when paired with reranking, metadata

In [18]:
count_tokens(
    conversation_sum, 
    "Which data source types could be used to give context to the model?"
)

Spent a total of 2058 tokens


'A model can get context from many kinds of sources. The best choices depend on whether you need authoritative internal information, current public facts, structured data, or material the user provides.\n\n- **Documents and files:** PDFs, Word documents, slide decks, manuals, policies, reports, emails, and spreadsheets. These are common sources for retrieval-augmented generation (RAG).\n- **Databases:** Relational databases, data warehouses, and other structured stores. Useful for precise facts, records, and calculations—for example, inventory levels or account details.\n- **APIs and live services:** Search engines, weather or financial services, product catalogs, and internal business systems. These can provide fresh information or let the model look up specific records.\n- **Knowledge bases and knowledge graphs:** Wikis, glossaries, ontologies, and graphs that represent entities and their relationships. Helpful when the connections between things matter.\n- **Web pages and public dat

In [19]:
count_tokens(
    conversation_sum, 
    "What is my aim again?"
)

Spent a total of 1514 tokens


'Your aim is to explore how Large Language Models can use external knowledge, and to compare the different approaches and data sources. You haven’t mentioned a specific research topic or application area yet.'

In this case the summary contains enough information for the LLM to “remember” our original aim. We can see this summary in it’s raw form like so:

In [20]:
print(conversation_sum.memory.buffer)

The human greets the AI with “Good morning.” The AI returns the greeting, wishes the human a lovely start to the day, and asks how it can help. The human wants to explore integrating Large Language Models with external knowledge. The AI explains that external knowledge can provide newer or specialized information and outlines retrieval-augmented generation (RAG), tools and APIs, knowledge graphs, and fine-tuning. It notes challenges such as finding reliable evidence, tracking provenance, resolving conflicts, and acknowledging when sources are insufficient, then asks whether the human has a research topic, system, or application area in mind.

The human wants to analyze the different possibilities. The AI compares RAG for changing or specialized document collections; live web search and APIs for current information; tools for carrying out actions, with stronger safeguards for consequential changes; knowledge graphs for relationships and dependencies; long-context input for a small suppl

The number of tokens being used for this conversation is greater than when using the `ConversationBufferMemory`, so is there any advantage to using `ConversationSummaryMemory` over the buffer memory?

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/token_interaction.webp" alt='auto' width="1000"/>
</div>

Token count (y-axis) for the buffer memory vs. summary memory as the number of interactions (x-axis) increases.

For longer conversations. As shown above, the summary memory initially uses far more tokens. However, as the conversation progresses, the summarization approach grows more slowly. In contrast, the buffer memory continues to grow linearly with the number of tokens in the chat.

We can summarize the pros and cons of ConversationSummaryMemory as follows:

We can summarize the pros and cons of `ConversationSummaryMemory` as follows:

<!DOCTYPE html>
<html>
<head>
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid black;
            padding: 8px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>

<table>
    <tr>
        <th>Pros</th>
        <th>Cons</th>
    </tr>
    <tr>
        <td>Shortens the number of tokens for long conversations.</td>
        <td>Can result in higher token usage for smaller conversations.</td>
    </tr>
    <tr>
        <td>Enables much longer conversations.</td>
        <td>Memorization of the conversation history is wholly reliant on the summarization ability of the intermediate summarization LLM.</td>
    </tr>
    <tr>
        <td>Relatively straightforward implementation, intuitively simple to understand.</td>
        <td>Also requires token usage for the summarization LLM; this increases costs (but does not limit conversation length).</td>
    </tr>
</table>

</body>
</html>


Conversation summarization is a good approach for cases where long conversations are expected. Yet, it is still fundamentally limited by token limits. After a certain amount of time, we still exceed context window limits - Maybe NOT, depending on the LLM being used.

### ConversationBufferWindowMemory
The `ConversationBufferWindowMemor`y acts in the same way as our earlier “buffer memory” but adds a window to the memory. Meaning that we only keep a given number of past interactions before “forgetting” them. We use it like so:

In [21]:
conversation_bufw = ConversationChain(
	llm=llm,
	memory=ConversationBufferWindowMemory(k=1)
)

In this instance, we set `k=1` — this means the window will remember the single latest interaction between the human and AI. That is the latest human response and the latest AI response. We can see the effect of this below:

In [22]:
count_tokens(
    conversation_bufw, 
    "Good morning AI!"
)

Spent a total of 137 tokens


'Good morning! Hope your day is off to a great start. What can I help you with today?'

In [23]:
count_tokens(
    conversation_bufw, 
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

Spent a total of 442 tokens


'That’s a fascinating area. A useful starting point is to distinguish a few ways an LLM can work with external knowledge:\n\n- **Retrieval-augmented generation (RAG):** The system searches documents or a database for relevant material, then gives that material to the model as context for its answer. This is often a good fit for knowledge that changes over time or needs to be traceable to sources.\n- **Tool use:** The model calls an external system—such as a search engine, calculator, API, or database—when it needs current information or must take an action.\n- **Fine-tuning:** The model is trained further on examples or domain-specific material. This can shape its behavior or terminology, though it’s generally less suitable than retrieval for keeping a changing knowledge base up to date.\n- **Knowledge graphs or structured databases:** These can represent entities and relationships explicitly, which can help with precise queries and multi-step reasoning.\n\nThe right choice depends on 

In [24]:
count_tokens(
    conversation_bufw, 
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 1217 tokens


'There are several ways to think about the design space. The main question is not just *how the model gets information*, but also *what kind of information it needs, when it needs it, and how the answer should be checked*.\n\n### Ways to connect the model to knowledge\n\n- **Search and RAG:** Search a collection of documents, then provide the most relevant passages to the model. This works well for manuals, policies, research papers, and organizational knowledge. It can support citations, though retrieval and citation quality need to be evaluated separately.\n- **Structured data access:** Let the model query databases, knowledge graphs, spreadsheets, or APIs. This is useful when questions involve precise facts, filters, relationships, or calculations—for example, “Which products are available in region X?”\n- **Live tools:** Give the model access to search engines, calculators, code execution, or operational services. This is a natural fit for current information or tasks that involve 

In [25]:
count_tokens(
    conversation_bufw, 
    "Which data source types could be used to give context to the model?"
)

Spent a total of 1305 tokens


'A model’s context can come from many kinds of sources, depending on the task:\n\n- **The conversation itself:** The user’s question, earlier messages, and anything they paste or attach.\n- **Documents and files:** PDFs, manuals, policies, reports, emails, spreadsheets, presentations, or code repositories. These can be supplied directly or searched with retrieval (RAG).\n- **Structured databases:** Tables in SQL databases, data warehouses, spreadsheets, or other systems—useful for precise facts, filters, and calculations.\n- **APIs and live services:** Current information from products, business systems, calendars, maps, weather services, or other applications.\n- **Web sources:** Search results and pages from the public web, or from a private company intranet.\n- **Knowledge graphs and directories:** Entities and their relationships, such as people, products, organizations, and how they connect.\n- **Past interactions or saved memory:** User preferences or details from earlier convers

In [26]:
count_tokens(
    conversation_bufw, 
    "What is my aim again?"
)

Spent a total of 563 tokens


'From this conversation, your aim seems to be understanding which kinds of data sources can provide context to a model. You haven’t mentioned a broader goal yet.'

By the end of the conversation, when we ask **"What is my aim again?"**, the answer to this was contained in the human response three interactions ago. As we only kept the most recent interaction (`k=1`), the model had forgotten and could not give the correct answer.

We can see the effective “memory” of the model like so:

In [27]:
bufw_history = conversation_bufw.memory.load_memory_variables(
    inputs=[]
)['history']

In [28]:
print(bufw_history)

Human: What is my aim again?
AI: From this conversation, your aim seems to be understanding which kinds of data sources can provide context to a model. You haven’t mentioned a broader goal yet.


Although this method isn’t suitable for remembering distant interactions, it is good at limiting the number of tokens being used — a number that we can increase/decrease depending on our needs. For the longer conversation used in our earlier comparison, we can set `k=6` and reach ~1.5K tokens per interaction after 27 total interactions:

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/conversation_bw.webp" alt='auto' width="1000"/>
</div>

Token count including the ConversationBufferWindowMemory at k=6 and k=12.

If we only need memory of recent interactions, this is a great option. However, for a mix of both distant and recent interactions, there are other options.

### ConversationSummaryBufferMemory
The `ConversationSummaryBufferMemory` is a mix of the `ConversationSummaryMemory` and the `ConversationBufferWindowMemory`. It summarizes the earliest interactions in a conversation while maintaining the max_token_limit most recent tokens in their conversation. It is initialized like so:

In [29]:
conversation_sum_bufw = ConversationChain(
    llm=llm, memory=ConversationSummaryBufferMemory(
        llm=llm,
        max_token_limit=650
))

When applying this to our earlier conversation, we can set `max_token_limit` to a small number and yet the LLM can remember our earlier “aim”.

This is because that information is captured by the “summarization” component of the memory, despite being missed by the “buffer window” component.

Naturally, the pros and cons of this component are a mix of the earlier components on which this is based.

<!DOCTYPE html>
<html>
<head>
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid black;
            padding: 10px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>

<table>
    <tr>
        <th>Pros</th>
        <th>Cons</th>
    </tr>
    <tr>
        <td>Summarizer means we can remember distant interactions</td>
        <td>Summarizer increases token count for shorter conversations</td>
    </tr>
    <tr>
        <td>Buffer prevents us from missing information from the most recent interactions</td>
        <td>Storing the raw interactions — even if just the most recent interactions — increases token count</td>
    </tr>
</table>

</body>
</html>


Although requiring more tweaking on what to summarize and what to maintain within the buffer window, the `ConversationSummaryBufferMemory` does give us plenty of flexibility and is the only one of our memory types (so far) that allows us to remember distant interactions and store the most recent interactions in their raw — and most information-rich — form.

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/memory_bws.webp" alt='auto' width="1000"/>
</div>

Token count comparisons including the ConversationSummaryBufferMemory type with max_token_limit values of 650 and 1300.

We can also see that despite including a summary of past interactions and the raw form of recent interactions — the increase in token count of `ConversationSummaryBufferMemory` is competitive with other methods.

### Other Memory Types
The memory types we have covered here are great for getting started and give a good balance between remembering as much as possible and minimizing tokens.

However, we have other options — particularly the `ConversationKnowledgeGraphMemory` and `ConversationEntityMemory`. We’ll give these different forms of memory the attention they deserve in upcoming chapters.

That’s it for this introduction to conversational memory for LLMs using LangChain. As we’ve seen, there are plenty of options for helping stateless LLMs interact as if they were in a stateful environment — able to consider and refer back to past interactions.

As mentioned, there are other forms of memory we can cover. We can also implement our own memory modules, use multiple types of memory within the same chain, combine them with agents, and much more. All of which we will cover in the future.

<br>


## Main Memory Types

| Memory Type | How it works | Pros | Cons | Best for |
|-------------|--------------|------|------|----------|
| **ConversationBufferMemory** | Stores the entire conversation. | Simple, maximum context. | Token usage grows continuously; higher cost; eventually exceeds context window. | Short conversations. |
| **ConversationBufferWindowMemory** | Stores only the last **k** interactions. | Predictable token usage; inexpensive. | Older context is forgotten. | Medium-length chats. |
| **ConversationSummaryMemory** | Uses an LLM to summarize previous conversations. | Much lower token growth over long conversations. | Summary may omit details; requires extra LLM calls. | Long conversations. |
| **ConversationSummaryBufferMemory** | Keeps recent messages + summarizes older ones. | Good balance between detail and efficiency. | More complex than other approaches. | Most production chatbots. |

